In [1]:
from beir.datasets.data_loader import GenericDataLoader
from helpers.converter import convert_corpus_jsonl
import os

/Users/dieudonne/Developer/dense-sparse/.venv/lib/python3.13/site-packages/beir/datasets/data_loader.py:8: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from tqdm.autonotebook import tqdm


In [ ]:
## necessary directories
data_path = "./datasets/nfcorpus/" ## For the normal docs
index_path = "datasets/index/" ## For the inverted index
output_doc_path = "./datasets/docs" ## for the converted docs 
qrel_file = "./datasets/nfcorpus/qrels/test.tsv"
splade_index_path = "datasets/splade-index"
splade_model = 'naver/splade-cocondenser-ensembledistil' 
splade_path = "./datasets/docs/splade"


In [3]:
## Load the dataset
corpus, queries, qrels = GenericDataLoader(data_path).load()

100%|██████████| 3633/3633 [00:00<00:00, 121030.86it/s]


In [4]:
## Convert the dataset to jsonl
new_data_path = convert_corpus_jsonl(corpus=corpus, output_path=output_doc_path)

Saved 3633 documents to ./datasets/docs/corpus.jsonl


In [5]:
from sparse.bm25 import BM25Retrieval
from sparse.splade import Splade

# Create the index folder if not exist
os.makedirs(index_path, exist_ok=True)

bm25 = BM25Retrieval(index_path) ## We create the index it does not exist
bm25.index(new_data_path) ## Index the data

splade = Splade(splade_index_path, splade_model)

apr 17, 2026 11:00:45 PM org.apache.lucene.store.MemorySegmentIndexInputProvider <init>
INFORMAZIONI: Using MemorySegmentIndexInput with Java 21; to disable start with -Dorg.apache.lucene.store.MMapDirectory.enableMemorySegments=false
Loading weights: 100%|██████████| 204/204 [00:00<00:00, 33751.65it/s]
BertForMaskedLM LOAD REPORT from: naver/splade-cocondenser-ensembledistil
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [6]:
## Evaluate bm25
results = {}

for qid, query in queries.items():
    hits = bm25.searcher.search(query, k=10)
    
    results[qid] = {}
    for hit in hits:
        results[qid][hit.docid] = hit.score
    
    

In [7]:
## Tokenise and encode the splade model
tokens = splade.convert_corpus(f"{new_data_path}/corpus.jsonl", "splade_vectors.jsonl")

In [14]:
## Create the inverted index for splade
splade.index(f"{new_data_path}/splade")

Loading weights: 100%|██████████| 204/204 [00:00<00:00, 25996.96it/s]
BertForMaskedLM LOAD REPORT from: naver/splade-cocondenser-ensembledistil
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [16]:
splade_results = {}
for qid, query in queries.items():
    hits = splade.search(query)  # use your existing method
    splade_results[qid] = {}
    for hit in hits:
        splade_results[qid][hit["docid"]] = hit["score"]

In [17]:
splade_results

{'PLAIN-2': {'MED-2429': 701.0,
  'MED-2431': 675.0,
  'MED-4827': 574.0,
  'MED-10': 570.0,
  'MED-14': 570.0,
  'MED-1732': 552.0,
  'MED-2427': 551.0,
  'MED-2440': 551.0,
  'MED-2434': 528.0,
  'MED-2122': 523.0},
 'PLAIN-12': {'MED-4711': 431.0,
  'MED-2603': 309.0,
  'MED-2501': 307.0,
  'MED-4113': 263.0,
  'MED-3317': 255.0,
  'MED-1116': 248.0,
  'MED-2028': 248.0,
  'MED-3706': 248.0,
  'MED-4114': 248.0,
  'MED-4115': 248.0},
 'PLAIN-23': {'MED-1961': 436.0,
  'MED-1169': 421.0,
  'MED-2661': 402.0,
  'MED-118': 393.0,
  'MED-2651': 393.0,
  'MED-2658': 375.0,
  'MED-2652': 360.0,
  'MED-2495': 355.0,
  'MED-4505': 354.0,
  'MED-1271': 339.0},
 'PLAIN-33': {'MED-2721': 588.0,
  'MED-4763': 545.0,
  'MED-3058': 501.0,
  'MED-2717': 499.0,
  'MED-1803': 495.0,
  'MED-2722': 493.0,
  'MED-2723': 489.0,
  'MED-5006': 486.0,
  'MED-1468': 472.0,
  'MED-1797': 467.0},
 'PLAIN-44': {'MED-2811': 516.0,
  'MED-2817': 506.0,
  'MED-2831': 504.0,
  'MED-1948': 495.0,
  'MED-2814': 493.

In [18]:
### Evaluate the bm25
from evaluation.metrics import compute_ndcg

metrics = compute_ndcg(qrels, results) 
metrics_splade = compute_ndcg(qrels, splade_results)

In [19]:
metrics_splade

{'NDCG@1': 0.43344, 'NDCG@10': 0.31908, 'NDCG@100': 0.20898}

In [13]:
metrics

{'NDCG@1': 0.4257, 'NDCG@10': 0.32178, 'NDCG@100': 0.20768}